# 01B 亲手检查微型 Transformer
对应 **L01.05**。用矩阵、形状和“修改未来字符”的测试理解因果注意力。

本 Notebook 展示完整模型代码：字符嵌入、位置嵌入、一层注意力、一层前馈网络、残差、归一化和词表投影。只做前向与结构检查；训练留到 01C。


## 学习路线与配套课件

实践 1B：因果掩码与模型结构（`L01-P02`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L01.05-S01、L01.05-S02、L01.05-S03、L01.05-S04、L01.05-S05、L01.05-S06、L01.05-S07。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys
# 无论从仓库根目录还是 Notebook 目录启动，都定位到同一份课程资料。
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "01 大模型基础与最小训练实验"
import torch
from torch.nn import functional as F
torch.manual_seed(7)
torch.set_num_threads(2)
print("PyTorch:", torch.__version__, "；本实验使用 CPU")
from torch import nn
import math


PyTorch: 2.8.0+cpu ；本实验使用 CPU


## Token ID 变成向量
ID 只是索引；embedding 是可训练的查表结果。绝对位置 embedding 告诉模型每个字符在窗口中的位置。

**先预测：**同一个字符出现在两个位置时，Token embedding 相同吗？加上位置向量后还相同吗？


In [2]:
token = nn.Embedding(10, 4)
position = nn.Embedding(8, 4)
ids = torch.tensor([[3,3,5]])
embedded = token(ids)
x = embedded + position(torch.arange(ids.shape[1]))
assert torch.equal(embedded[0,0], embedded[0,1])
print("Token 向量相同:", torch.equal(embedded[0,0],embedded[0,1]))
print("加位置后相同:", torch.equal(x[0,0],x[0,1]), "形状:", x.shape)


Token 向量相同: True
加位置后相同: False 形状: torch.Size([1, 3, 4])


## 当前位置能看见谁
行是查询位置，列是被读取的位置。第 i 行只能读取 j ≤ i 的位置。这里把所有原始分数设为 0，便于只观察掩码的作用。

**先预测：**第一个位置会给第 4 个位置多少注意力？


In [3]:
mask = torch.tril(torch.ones(4,4,dtype=torch.bool))
scores = torch.zeros(4,4)
weights = scores.masked_fill(~mask,float("-inf")).softmax(-1)
print("可见位置（1 可见）:\n", mask.int())
print("掩码后的注意力:\n", weights)
assert torch.allclose(weights.sum(-1), torch.ones(4))
assert torch.count_nonzero(weights.triu(1)) == 0


可见位置（1 可见）:
 tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
掩码后的注意力:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])


## Q、K、V 和多头
Q 与 K 计算匹配分数，除以 √D 调节尺度；掩码之后 softmax，再用权重对 V 加权。注意力权重是本次计算中的系数，不能直接当作完整的因果解释。

下面是可训练模型使用的完整实现。重点追踪 [B,T,C] 与 [B,H,T,D]，不要求记住每个转置。


### 缩放点积注意力：公式与小算例

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V.$$

允许位置 $M=0$，禁止位置 $M=-\infty$。一个查询的缩放分数为 `[2,1,9]`，第三位置被屏蔽后权重约为 `[0.7311,0.2689,0]`。取一维 $V=[10,20,999]$，加权输出约为 $12.6894$（使用未舍入权重）。

先按三个动作读公式：Q 与 K 的转置相乘得到匹配分数，除以键维度的平方根控制尺度；加掩码后做 softmax，得到每个可见位置的权重；最后用权重对 V 加权求和。M 在允许位置为 0，在禁止位置为负无穷。本页只看一个查询，并用一维 V 简化手算，省略 dropout。三个缩放后的分数假设为 2、1、9；第三个位置是未来位置，即使分数最大，也先被屏蔽。剩下两个分数经过 softmax，权重约为 0.7311、0.2689。设三个值为 10、20、999，输出约 12.6894，而不是直接选择 10，也不会读入 999。Q、K、V 在模型中是向量，这个标量小例子只是解释加权求和。请预测把 999 改成更大的数会不会影响输出；下方会先验证这个局部算例，再检查完整模型的因果性。

参考：https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.scaled_dot_product_attention.html


In [4]:
class CausalAttention(nn.Module):
    def __init__(self, width, heads, context):
        super().__init__()
        assert width % heads == 0
        self.heads, self.head_dim = heads, width // heads
        self.qkv = nn.Linear(width, 3 * width)
        self.out = nn.Linear(width, width)
        self.register_buffer("mask", torch.tril(torch.ones(context, context, dtype=torch.bool)))

    def forward(self, x):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # [B,T,C] -> [B,H,T,D]，每个头学习不同的关系。
        q, k, v = [t.reshape(batch, length, self.heads, self.head_dim).transpose(1, 2)
                   for t in (q, k, v)]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.mask[:length, :length], float("-inf"))
        weights = scores.softmax(dim=-1)
        mixed = (weights @ v).transpose(1, 2).contiguous().reshape(batch, length, width)
        return self.out(mixed)


### 算出一次被掩码的注意力

配套课件：L01.05-S03、L01.05-S04。

把 $QK^T/\sqrt{d_k}$ 已算出的分数设为 `[2, 1, 9]`。当前位置只能看前两项，未来项即使分数最高，也必须先置为 $-\infty$，再做 softmax。为了看清加权求和，先把每个 V 简化为一个数 `[10, 20, 999]`。实际多头注意力中的 V 是向量。

In [5]:
case_scores = torch.tensor([2., 1., 9.])
case_visible = torch.tensor([True, True, False])
case_masked = case_scores.masked_fill(~case_visible, float("-inf"))
case_weights = torch.softmax(case_masked, dim=-1)
case_values = torch.tensor([10., 20., 999.])
case_output = case_weights @ case_values
print("掩码后分数:", case_masked.tolist())
print("权重:", [round(v, 4) for v in case_weights.tolist()])
print("加权结果:", round(case_output.item(), 4))
assert case_weights[2] == 0
assert torch.isclose(case_output, torch.tensor(12.6894), atol=1e-4)
# 只改未来的 V，当前输出仍相同。
case_changed_values = torch.tensor([10., 20., -999.])
assert torch.equal(case_weights @ case_changed_values, case_output)


掩码后分数: [2.0, 1.0, -inf]
权重: [0.7311, 0.2689, 0.0]
加权结果: 12.6894


**结果解读**

权重约为 `[0.7311, 0.2689, 0]`，结果约为 12.6894。未来位置的 V 没有贡献；这一步只是注意力局部算例，后面的完整模型实验才检查整个网络的因果性。

**小练习**

把掩码全设为 True，观察未来分数 9 怎样控制结果。解释为什么“把未来权重设小一点”不能替代严格掩码。

<details><summary>完成后展开参考分析</summary>

不掩码时未来权重接近 1，输出被 999 主导。只要权重非零，未来信息就可能进入当前输出。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 残差、归一化与前馈层
本课采用预归一化结构。残差把输入加回输出；LayerNorm 与注意力、前馈层各自承担不同工作。前馈层逐位置变换特征，注意力负责混合不同位置的信息。


In [6]:
class Block(nn.Module):
    def __init__(self, width, heads, context):
        super().__init__()
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.attention = CausalAttention(width, heads, context)
        self.ffn = nn.Sequential(nn.Linear(width, 4 * width), nn.GELU(), nn.Linear(4 * width, width))

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.ffn(self.norm2(x))


### 残差保留什么，LayerNorm 归一化什么

配套课件：L01.05-S05、L01.05-S06。

本课采用预归一化：$h=x+\mathrm{Attention}(\mathrm{LN}(x))$，$y=h+\mathrm{FFN}(\mathrm{LN}(h))$。残差支路做逐元素相加；LayerNorm 在每个位置的特征维上计算均值和方差。下面先用两维数值把两种操作拆开，再观察前馈层的形状。这里没有训练参数。

In [7]:
case_x = torch.tensor([1., 3.])
case_delta = torch.tensor([0.2, -0.1])
case_h = case_x + case_delta
case_mean = case_x.mean()
case_var = case_x.var(unbiased=False)  # LayerNorm 使用总体方差
case_ln = (case_x - case_mean) / torch.sqrt(case_var + 1e-5)
print("原输入:", case_x.tolist(), "新信息:", [round(v, 4) for v in case_delta.tolist()])
print("残差相加:", [round(v, 4) for v in case_h.tolist()])
print("均值 / 方差:", case_mean.item(), case_var.item(), "归一化:", case_ln.tolist())
assert torch.allclose(case_ln, F.layer_norm(case_x, (2,)))
# 隔离随机数，避免改变后面完整模型的初始化与对照。
with torch.random.fork_rng():
    case_ffn = nn.Sequential(nn.Linear(48, 192), nn.GELU(), nn.Linear(192, 48))
    case_batch = torch.zeros(2, 4, 48)
    print("FFN:", tuple(case_batch.shape), "→", tuple(case_ffn(case_batch).shape))
    assert case_ffn(case_batch).shape == case_batch.shape


原输入: [1.0, 3.0] 新信息: [0.2, -0.1]
残差相加: [1.2, 2.9]
均值 / 方差: 2.0 1.0 归一化: [-0.9999949932098389, 0.9999949932098389]
FFN: (2, 4, 48) → (2, 4, 48)


**结果解读**

残差结果为 `[1.2, 2.9]`；归一化约为 `[-1, 1]`。FFN 将每个位置的 48 维特征扩展到 192 维再投回 48 维，位置数 4 不变，因此能与残差支路相加。LayerNorm 还可带可学习的缩放与偏置，本算例使用默认 1 和 0。

**小练习**

改成输入 `[11, 13]`：LayerNorm 结果会变吗？残差相加结果会变吗？然后指出 Block.forward 中哪一条路径保留了原输入。

<details><summary>完成后展开参考分析</summary>

共同平移被均值减法消去，所以归一化结果不变；残差结果会共同增加 10。原输入由加号左边的 x 或 h 保留。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 把模块连成语言模型
输出形状是 [B,T,V]。一个样本中的每个位置都输出词表大小的一组 logits。模型不在 forward 里采样；训练程序计算 Loss，推理程序选择 Token。


In [8]:
class TinyLM(nn.Module):
    def __init__(self, vocab_size, width=48, heads=4, context=32):
        super().__init__()
        self.context = context
        self.token_embedding = nn.Embedding(vocab_size, width)
        self.position_embedding = nn.Embedding(context, width)
        self.block = Block(width, heads, context)
        self.norm = nn.LayerNorm(width)
        self.output = nn.Linear(width, vocab_size)

    def forward(self, ids):
        length = ids.shape[1]
        if not 0 < length <= self.context:
            raise ValueError("输入长度必须在上下文窗口内")
        pos = torch.arange(length, device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(pos)
        return self.output(self.norm(self.block(x)))  # [B,T,V]，返回 logits


In [9]:
model = TinyLM(vocab_size=20, width=48, heads=4, context=8)
ids = torch.tensor([[1,2,3,4,5,6], [2,3,4,5,6,7]])
logits = model(ids)
print("输入:", ids.shape, "输出:", logits.shape)
print("参数量:", sum(p.numel() for p in model.parameters()))
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))
assert logits.shape == (2,6,20)


输入: torch.Size([2, 6]) 输出: torch.Size([2, 6, 20])
参数量: 30692
token_embedding.weight (20, 48)
position_embedding.weight (8, 48)
block.norm1.weight (48,)
block.norm1.bias (48,)
block.norm2.weight (48,)
block.norm2.bias (48,)
block.attention.qkv.weight (144, 48)
block.attention.qkv.bias (144,)
block.attention.out.weight (48, 48)
block.attention.out.bias (48,)
block.ffn.0.weight (192, 48)
block.ffn.0.bias (192,)
block.ffn.2.weight (48, 192)
block.ffn.2.bias (48,)
norm.weight (48,)
norm.bias (48,)
output.weight (20, 48)
output.bias (20,)


## 修改未来，过去的输出应该不变
**先预测：**只替换第 4 个及以后的输入，第 1–3 个位置的 logits 会变化吗？这个测试比“图上画了一个三角形”更能检查实现。


In [10]:
model.eval()
original = torch.tensor([[1,2,3,4,5,6]])
changed = torch.tensor([[1,2,3,9,9,9]])
with torch.no_grad():
    a, b = model(original), model(changed)
difference = (a[:,:3] - b[:,:3]).abs().max().item()
print("前三个位置最大差异:", difference)
assert torch.allclose(a[:,:3], b[:,:3], atol=1e-6)


前三个位置最大差异: 0.0


### 练习：故意去掉因果限制
复制模型，把 mask 全部设成 True，再比较上面的两个输入。不要修改原模型，以免后续使用了错误结构。


In [11]:
import copy
student_model = copy.deepcopy(model)
# TODO：修改 student_model.block.attention.mask，然后计算前三个位置输出差异。


In [12]:
noncausal = copy.deepcopy(model)
noncausal.block.attention.mask.fill_(True)
with torch.no_grad():
    delta = (noncausal(original)[:,:3] - noncausal(changed)[:,:3]).abs().max().item()
print("允许读取未来后的最大差异:", delta)
assert delta > 1e-6
# 训练时若能读取标签所在的未来位置，Loss 可能很好看，但生成时没有这些未来字符。


允许读取未来后的最大差异: 0.1479804515838623


## 提交与自检
提交正常模型和去掉掩码后的差异数值，画出自己的 [B,T] → [B,T,C] → [B,T,V] 说明。

解释：注意力负责混合哪些维度？为什么输出需要覆盖整个词表？为什么我们的第一讲模型还不能胜任指令问答？

接着运行 **03-train-and-generate.ipynb**。
